<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/06_capstone_rf_vs_sjepa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - Capstone: Random Forest vs S-JEPA, on identical folds, with controls

This is the scientific payoff, and it comes with a result that is honest rather than flattering. We put several systems side by side on the **same videos**, the **same locked leakage-safe folds**, and the **same pooled out-of-fold scoring**:

1. a classical **Random Forest** on hand-made gait features (the exp5 recipe, three classes),
2. the **label-free S-JEPA** with a frozen linear probe,
3. cheap **shortcut controls** (visibility, body size, static pose) that any real representation must beat before we trust it.

The headline is **pooled macro-F1** over the folds (one prediction per clip, gathered across all held-out folds), because averaging per-fold F1 on ~9 test videos is even noisier. We report it beside the paired RF and the controls, then say plainly what it does and does not mean.

> **Spoiler, stated up front.** On this tiny, already-inspected, source-grouped collection the S-JEPA scores *below* both the Random Forest and the nuisance controls. That is the expected, plan-anticipated outcome of removing the shortcuts and the label leak that inflated an earlier number, and it is reported as a negative result, not hidden.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'rf_vs_sjepa.svg')))
display(SVG(filename=str(IMAGES_DIR / 'grouped_split.svg')))
display(SVG(filename=str(IMAGES_DIR / 'eval_firewall.svg')))

## The frozen result is produced by a script, not the notebook

So the headline cannot drift as someone re-runs cells, the authoritative R1 run lives in `scripts/scripts_r1_repaired.py` and its output is committed under `artifacts/runs/r1_g1_1k_s42/`. That script and this notebook share the identical fold registry, so the comparison is paired by construction. We read the frozen numbers first, then reproduce the mechanism live at a smaller budget so you can see how it is built.


In [ ]:
import json
frozen_path = ARTIFACT_DIR / 'runs' / 'r1_g1_1k_s42' / 'results.json'
if frozen_path.exists():
    frozen = json.loads(frozen_path.read_text())
    sj = frozen['sjepa_pooled']; rf = frozen['rf_pooled']
    print('Frozen R1 (1000 updates, seed 42, all 5 folds, pooled OOF):')
    print(f"  S-JEPA          : macro-F1 {sj['macro_f1']:.3f} | acc {sj['accuracy']:.3f}"
          f" | PD-recall {sj['pd_recall']:.3f}")
    print(f"  Random Forest   : macro-F1 {rf['macro_f1']:.3f} | acc {rf['accuracy']:.3f}"
          f" | PD-recall {rf['pd_recall']:.3f}")
    print('  effective rank per fold:',
          [round(d['eff_rank_final'], 1) for d in frozen['diagnostics']],
          '(all >> 1 -> no collapse)')
else:
    print('Frozen run not found. Reproduce it with:')
    print('  python scripts/scripts_r1_repaired.py --total-updates 1000 --seed 42 \\')
    print('      --output-dir artifacts/runs/r1_g1_1k_s42')

## Shortcut controls: the bar S-JEPA has to clear

Phase 0 also scored cheap nuisance features on the identical folds. If a control matches or beats S-JEPA, then S-JEPA is not yet using gait beyond what a camera artifact already reveals. We read those frozen control scores here.


In [ ]:
e0_path = ARTIFACT_DIR / 'eval' / 'g1' / 'E0_results.json'
if e0_path.exists():
    e0 = json.loads(e0_path.read_text())
    # Use POOLED macro-F1 for the controls so they are the SAME metric as the
    # pooled S-JEPA/RF above (pooling one prediction per clip across folds).
    # Averaging per-fold macro-F1 is a different, non-comparable number.
    print('Shortcut controls on g1 (best of logreg/rf, pooled OOF macro-F1):')
    for name, res in e0['shortcut_controls'].items():
        best = max(res['logreg']['pooled_macro_f1'], res['rf']['pooled_macro_f1'])
        print(f'  {name:16s}: {best:.3f}')
    print(f"E0 Random Forest pooled macro-F1: {e0['E0_RF']['pooled_macro_f1']:.3f}")
else:
    print('Run scripts/scripts_phase0_provenance.py to generate the control table.')

## Reproduce the mechanism live (one fold, small budget)

Now the moving parts, so nothing is a black box. For one locked fold we run the exact pipeline: paired RF, then **label-free** S-JEPA (no diagnosis label enters the objective), a frozen mean-pool over a fixed read-out, and a class-balanced probe fit on the training clips only. This uses a tiny update budget to stay fast; the frozen numbers above are the ones to cite.


In [ ]:
import numpy as np, torch, os
from sjepa.config import get_config
from sjepa.models import build_model, pick_device
from sjepa.train_v2 import train_sjepa_v2
from sjepa.masking_v2 import sample_target_mask
from sjepa.data import load_index, SequenceWindowDataset, sliding_windows
from sjepa.classical import build_feature_matrix, train_rf_and_predict
from sjepa.eval import evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

cfg = get_config(); device = pick_device()
LABELS = ['normal', 'ms', 'pd']
records = load_index(KEYPOINTS_DIR)
by_clip = {r.clip_name: r for r in records}
registry = json.loads((ARTIFACT_DIR / 'eval' / 'g1' / 'fold_registry.json').read_text())
readout = sample_target_mask(cfg.num_joints, cfg.num_time_tokens,
                             np.random.default_rng(0), target_ratio=0.6)
tm = torch.from_numpy(readout).to(device)

def embed_records(model, recs):
    V, Y = [], []
    for r in recs:
        w = sliding_windows(r.load_norm(), cfg.window_frames, cfg.window_stride)
        x = torch.from_numpy(w).float().to(device)
        with torch.no_grad():
            V.append(model.embed(x, tm).mean(0).cpu().numpy())
        Y.append(r.label)
    return np.stack(V), Y

In [ ]:
fold0 = registry['folds'][0]
train_recs = [by_clip[c] for c in fold0['train_clips']]
test_recs  = [by_clip[c] for c in fold0['test_clips']]

# paired Random Forest (exp5 recipe) on this fold
Xtr, ytr, _, _ = build_feature_matrix(train_recs, fps=cfg.target_fps)
Xte, yte, _, _ = build_feature_matrix(test_recs, fps=cfg.target_fps)
rf_pred = train_rf_and_predict(Xtr, ytr, Xte, seed=cfg.seed)
rf_m = evaluate(yte, rf_pred, LABELS)

# label-free S-JEPA on this fold's training sources
SMOKE = cfg.profile.endswith('smoke')  # correct parse of SJEPA_SMOKE (not a raw truthiness test)
UPDATES = 60 if SMOKE else 500
model = build_model(cfg, device=device, repaired=True)
ds = SequenceWindowDataset(train_recs, cfg.window_frames, cfg.window_stride)
state = train_sjepa_v2(model, ds, cfg, total_updates=UPDATES, device=device, mask_ratio=0.6)
Etr, ytr2 = embed_records(model, train_recs)
Ete, yte2 = embed_records(model, test_recs)
sc = StandardScaler().fit(Etr)                    # TRAIN only
probe = LogisticRegression(max_iter=2000, class_weight='balanced').fit(sc.transform(Etr), ytr2)
sj_m = evaluate(yte2, probe.predict(sc.transform(Ete)), LABELS)
print(f'live fold-0 demo (budget={UPDATES} updates): RF f1={rf_m.macro_f1:.3f}',
      f'| S-JEPA f1={sj_m.macro_f1:.3f} | eff_rank={state.eff_rank[-1]:.1f}')
print('(The frozen 5-fold pooled numbers above are the ones to cite, not this one fold.)')

## Confusion of the frozen S-JEPA run

Where does S-JEPA confuse the conditions across all held-out clips? The dominant error is PD read as MS, the same failure the Random Forest also struggles with here.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
if frozen_path.exists():
    fig, ax = plt.subplots(1, 2, figsize=(10,4))
    for a, key, title in [(ax[0], 'rf_pooled', 'Random Forest (pooled OOF)'),
                          (ax[1], 'sjepa_pooled', 'S-JEPA (pooled OOF)')]:
        cm = np.array(frozen[key]['confusion'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=LABELS, yticklabels=LABELS, ax=a)
        a.set_title(title); a.set_xlabel('predicted'); a.set_ylabel('true')
    plt.tight_layout(); plt.show()
else:
    print('Frozen run not found; run scripts/scripts_r1_repaired.py first.')

## A combined scoreboard

One table, every system on the identical g1 folds. This is the whole comparison in one place.


In [ ]:
import pandas as pd
rows = []
if frozen_path.exists():
    rows.append(('Random Forest (paired)', frozen['rf_pooled']['macro_f1']))
    rows.append(('S-JEPA (R1, 1k updates)', frozen['sjepa_pooled']['macro_f1']))
if e0_path.exists():
    for name, res in e0['shortcut_controls'].items():
        best = max(res['logreg']['pooled_macro_f1'], res['rf']['pooled_macro_f1'])
        rows.append((f'control: {name}', best))
rows.append(('chance (3 classes)', 1/3))
board = pd.DataFrame(rows, columns=['system', 'macro_F1']).sort_values('macro_F1', ascending=False)
display(board.reset_index(drop=True))
results = {'frozen_run': str(frozen_path.relative_to(ARTIFACT_DIR)) if frozen_path.exists() else None,
           'scoreboard': {n: float(v) for n, v in rows}}
(ARTIFACT_DIR / 'capstone_results.json').write_text(json.dumps(results, indent=2))
print('saved capstone_results.json')

## What this does and does not show

**What we can say.** The comparison is fair by construction: RF, S-JEPA, and the controls all sit on the identical locked folds, use per-clip pooled out-of-fold scoring, and fit their scalers and heads on training data only. On this data the Random Forest is the strongest system, and the label-free S-JEPA sits below it *and* below cheap nuisance controls.

**Why that is progress, not failure.** An earlier S-JEPA number looked higher partly because it leaned on shortcuts (every MS clip was filmed at 60fps and square) and a label leak in the objective. Removing those lowered the honest score. The representation did not collapse (effective rank stays well above 1 on every fold), so this is a real, non-degenerate estimate, not a broken run. Per the pre-registered rule, a mechanically valid model that does not clear the bar tells us to **stop scaling the local network** and fix the binding constraint instead.

**What we cannot say.** With ~47 videos from ~35 sources that are not verified people, this is a provisional, source-grouped **development estimate**, never a clinical result. No diagnostic, validity, or deployment claim is warranted.

**The honest next step.** The evidence points at the data pipeline and the acquisition domain, not model size: rebuild the lineage (true common frame rate, speed-preserving normalization, validity masks, a domain de-confound) and bring in external clinical-motion pretraining, then rerun. That completes the series: a skeleton pipeline from raw video, a label-free S-JEPA with stochastic clinically-guided masks, and a leakage-safe comparison that reports the result whichever way it falls.
